In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D

## Estilo de los plots

In [ ]:
# ─────────────────────────────────────────────
# 1. Funciones de estilo
# ─────────────────────────────────────────────
def config_ax_state(ax):
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)


def config_ax_IV(ax):
    # ax.grid(which="major", color="#DDDDDD", linewidth=0.8, zorder=-1)
    # ax.grid(which="minor", color="#DEDEDE", linestyle=":", linewidth=0.5, zorder=-1)
    ax.minorticks_on()
    # ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)


def setup_paper_plt(plt, latex=True, scaling: float = 1):
    plt.rcParams.update(
        {
            "pgf.texsystem": "pdflatex",
            "text.usetex": latex,
            "font.family": "mathpazo",
            "text.latex.preamble": "\n".join(
                [
                    r"\usepackage[utf8]{inputenc}",
                    r"\usepackage[T1]{fontenc}",
                    r"\usepackage{siunitx}",
                    r"\usepackage{physics}",
                ]
            ),
        }
    )
    BIGGER_SIZE = 11 * scaling
    BIGGEST_SIZE = 14 * scaling
    plt.rc("font", size=BIGGER_SIZE)
    plt.rc("axes", titlesize=BIGGER_SIZE)
    plt.rc("axes", labelsize=BIGGEST_SIZE)
    plt.rc("xtick", labelsize=BIGGEST_SIZE)
    plt.rc("ytick", labelsize=BIGGEST_SIZE)
    plt.rc("legend", fontsize=BIGGER_SIZE)
    plt.rc("figure", titlesize=BIGGEST_SIZE)


setup_paper_plt(plt, latex=True, scaling=2.5)

## Lectura Archivo de datos

In [ ]:
filename = "resultados_barrido_densidad.txt"
setup_paper_plt(plt, latex=True, scaling=2)

## V_reset vs Densidad

In [ ]:
output_name = "V_reset_vs_densidad.pdf"

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

diametros = sorted(df["Diametro (m)"].unique())
cmap = matplotlib.colormaps["plasma"].resampled(len(diametros))
color_map = {d: cmap(i) for i, d in enumerate(diametros)}

fig, ax = plt.subplots(figsize=(12, 9))

for d in sorted(diametros, reverse=True):
    sub = df[df["Diametro (m)"] == d]
    grp = sub.groupby("Dens.Vac.")["V_reset_abs"]
    media = grp.mean()
    std = grp.std()
    color = color_map[d]
    ax.plot(media.index, media.values, marker="o", color=color, label=f"{d * 1e9:.2f}", lw=1.8)
    ax.fill_between(media.index, media - std, media + std, color=color, alpha=0.15)

ax.set_xlabel(r"Vacancy density ($\text{vacancies}/\text{nm}^2$)")
ax.set_ylabel("|V$_{reset}$| (V)")
ax.set_title("V$_{reset}$ - Vacancy density by filament diameter")
ax.legend(title="Diam. (nm)", loc="upper right", fontsize=plt.rcParams["legend.fontsize"] * 0.75)
config_ax_IV(ax)
fig.tight_layout()
fig.savefig(output_name, dpi=150)
plt.close()

output_name = "V_reset_vs_densidad_sin_desviacion.pdf"

fig, ax = plt.subplots(figsize=(12, 9))

for d in sorted(diametros, reverse=True):
    sub = df[df["Diametro (m)"] == d]
    grp = sub.groupby("Dens.Vac.")["V_reset_abs"]
    media = grp.mean()
    color = color_map[d]
    ax.plot(media.index, media.values, marker="o", color=color, label=f"{d * 1e9:.2f}", lw=1.8)

ax.set_xlabel(r"Vacancy density ($\text{vacancies}/\text{nm}^2$)")
ax.set_ylabel("|V$_{reset}$| (V)")
ax.set_title("V$_{reset}$ - Vacancy density by filament diameter")
ax.legend(title="Diam. (nm)", loc="upper right", fontsize=plt.rcParams["legend.fontsize"] * 0.75)
config_ax_IV(ax)
fig.tight_layout()
fig.savefig(output_name, dpi=150)
plt.close()


## V_reset vs diametro

In [ ]:
output_name = "V_reset_vs_diametro.pdf"
# ---------------------------------------------------------------------------

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

densidades = sorted(df["Dens.Vac."].unique())
cmap = matplotlib.colormaps["viridis"].resampled(len(densidades))

fig, ax = plt.subplots(figsize=(14, 9))

for i, dens in enumerate(densidades):
    sub = df[df["Dens.Vac."] == dens]
    grp = sub.groupby("Diametro (m)")["V_reset_abs"]
    media = grp.mean()
    std = grp.std()
    color = cmap(i)
    ax.plot(
        media.index * 1e9,
        media.values,
        marker="s",
        color=color,
        label=f"{dens:.0f}",
        
        lw=1.8,
    )
    ax.fill_between(media.index * 1e9, media - std, media + std, color=color, alpha=0.10)

ax.set_xlabel("Filament diameter (nm)")
ax.set_ylabel("|V$_{reset}$| (V)")
ax.set_title("Mean V$_{reset}$Filament diameter by vacancy density")
ax.legend(title=r"Density ($\text{vac.}/\text{nm}^2$)", loc="upper left", ncol=1, bbox_to_anchor=(1.01, 1))
config_ax_IV(ax)

fig.tight_layout()
fig.savefig(output_name, dpi=150, bbox_inches="tight")
plt.close()

# ---------------------------------------------------------------------------
output_name = "V_reset_vs_diametro_sin_desviacion.pdf"

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

densidades = sorted(df["Dens.Vac."].unique())
cmap = matplotlib.colormaps["viridis"].resampled(len(densidades))

fig, ax = plt.subplots(figsize=(14, 9))

for i, dens in enumerate(densidades):
    sub = df[df["Dens.Vac."] == dens]
    grp = sub.groupby("Diametro (m)")["V_reset_abs"]
    media = grp.mean()
    std = grp.std()
    color = cmap(i)
    ax.plot(
        media.index * 1e9,
        media.values,
        marker="s",
        color=color,
        label=f"{dens:.0f}",
        lw=1.8,
    )

ax.set_xlabel("Filament diameter (nm)")
ax.set_ylabel("|V$_{reset}$| (V)")
ax.set_title("Mean V$_{reset}$  Filament diameter by vacancy density")
ax.legend(title=r"Density ($\text{vac.}/\text{nm}^2$)", loc="upper left", ncol=1, bbox_to_anchor=(1.01, 1))
config_ax_IV(ax)
fig.tight_layout()
fig.savefig(output_name, dpi=150, bbox_inches="tight")
plt.close()


## V_reset vs N_vecinos (1 por diámetro)

In [ ]:
vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
etiquetas = [r"$N_0$", r"$N_1$", r"$N_2$", r"$N_3$", r"$N_4$"]

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

diametros = sorted(df["Diametro (m)"].unique())
densidades = sorted(df["Dens.Vac."].unique())

cmap_vec = matplotlib.colormaps["viridis"].resampled(len(vecinos))
color_vec = {col: cmap_vec(i) for i, col in enumerate(vecinos)}

MARKERS = ["o", "D", "s", "^", "v", "P", "*", "X", "h", "<", ">", "p"]
marker_map = {dens: MARKERS[i % len(MARKERS)] for i, dens in enumerate(densidades)}

fs = plt.rcParams["legend.fontsize"]

leyenda_vec = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_vec[col], markersize=12, label=etiq)
    for col, etiq in zip(vecinos, etiquetas)
]
leyenda_dens = [
    Line2D([0], [0], marker=marker_map[dens], color="gray", markersize=8, linestyle="none", label=f"{int(dens)}")
    for dens in densidades
]

for d in diametros:
    sub = df[df["Diametro (m)"] == d]

    fig, ax = plt.subplots(figsize=(12, 9))

    for col in vecinos:
        for dens in densidades:
            s = sub[sub["Dens.Vac."] == dens]
            if s.empty:
                continue
            ax.scatter(
                s[col],
                s["V_reset_abs"],
                c=[color_vec[col]] * len(s),
                marker=marker_map[dens],
                alpha=0.7,
                edgecolors="none",
                s=100,
            )

    ax.set_xlabel("Number of vacancies")
    ax.set_ylabel("|V$_{reset}$| (V)")
    ax.set_title(f"|V$_{{reset}}$| vs vacancy count — diameter {d * 1e9:.2f} nm")

    leg1 = ax.legend(
        handles=leyenda_vec,
        title="Vacancy type",
        fontsize=fs * 0.8,
        title_fontsize=fs * 0.85,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
    )
    ax.add_artist(leg1)
    ax.legend(
        handles=leyenda_dens,
        title=r"Density (vac./nm$^2$)",
        fontsize=fs * 0.8,
        title_fontsize=fs * 0.85,
        loc="upper left",
        bbox_to_anchor=(1.01, 0.5),
        markerscale=1.2,
        handlelength=1.5,
        handletextpad=0.5,
    )

    config_ax_IV(ax)
    fig.tight_layout()
    fig.savefig(f"Vreset_vs_Nvecinos_{d * 1e9:.2f}nm.pdf", dpi=150, bbox_inches="tight")
    plt.close(fig)


## Compacidad del fiamento NO se usa

In [ ]:
# No se emplea

output_name = "Compactness_vs_V_reset.pdf"

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()
df["compacidad"] = (df["N_4"] + 1) / (df["N_0"] + 1)

diametros = sorted(df["Diametro (m)"].unique())
densidades = sorted(df["Dens.Vac."].unique())
cmap = matplotlib.colormaps["plasma"].resampled(len(diametros))
color_map = {d: cmap(i) for i, d in enumerate(diametros)}

MARKERS = ["o", "D", "s", "^", "v", "P", "*", "X", "h", "<", ">", "p"]
marker_map = {dens: MARKERS[i % len(MARKERS)] for i, dens in enumerate(densidades)}

fs = plt.rcParams["legend.fontsize"]

fig, ax = plt.subplots(figsize=(14, 9))

for d in diametros:
    for dens in densidades:
        sub = df[(df["Diametro (m)"] == d) & (df["Dens.Vac."] == dens)]
        if sub.empty:
            continue
        ax.scatter(
            sub["compacidad"],
            sub["V_reset_abs"],
            c=[color_map[d]] * len(sub),
            marker=marker_map[dens],
            s=175,
            alpha=1,
            edgecolors="none",
        )

mask = df["compacidad"].notna() & df["V_reset_abs"].notna()
x = df.loc[mask, "compacidad"].values
y = df.loc[mask, "V_reset_abs"].values
m, b = np.polyfit(np.log(x + 1e-9), y, 1)
r2 = np.corrcoef(np.log(x + 1e-9), y)[0, 1] ** 2
xfit = np.linspace(x.min(), x.max(), 300)
ax.plot(
    xfit,
    m * np.log(xfit + 1e-9) + b,
    color="black",
    lw=1.8,
    linestyle="--",
    label=f"$y={m:.3f}\\,\\ln(x){b:+.3f}$  ($R^2={r2:.3f}$)",
)

leg_fit = ax.legend(fontsize=fs * 0.9, loc="upper right")
ax.add_artist(leg_fit)

ax.set_xscale("log")

ax.set_xlabel(r"Filament compactness  $\frac{N_4+1}{N_0+1}$")
ax.set_ylabel("|V$_{reset}$| (V)")
ax.set_title("Filament compactness - V$_{reset}$ (color=diameter, marker=density)")
config_ax_IV(ax)

leyenda_color = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[d], markersize=8, label=f"{d * 1e9:.2f}")
    for d in diametros
]
leyenda_marker = [
    Line2D([0], [0], marker=marker_map[dens], color="gray", markersize=8, linestyle="none", label=f"{int(dens)}")
    for dens in densidades
]

leg1 = ax.legend(
    handles=leyenda_color,
    title="Diameter nm",
    fontsize=fs * 0.8,
    title_fontsize=fs * 0.85,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
)
ax.add_artist(leg1)

ax.legend(
    handles=leyenda_marker,
    title=r"Density (vac./nm$^2$)",
    fontsize=fs * 0.8,
    title_fontsize=fs * 0.85,
    loc="upper left",
    bbox_to_anchor=(1.01, 0.5),
    markerscale=1.2,
    handlelength=1.5,
    handletextpad=0.5,
)

fig.tight_layout()
fig.savefig(output_name, dpi=150, bbox_inches="tight")
plt.close()


## V_reset vs N_vecinos (1 por tipo de vacante) NO SE USA

In [ ]:
# No se va a emplear

vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
titulos = [
    r"$N_0$ — isolated vacancies",
    r"$N_1$ — 1 neighbour",
    r"$N_2$ — 2 neighbours",
    r"$N_3$ — 3 neighbours",
    r"$N_4$ — fully surrounded",
]

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

diametros = sorted(df["Diametro (m)"].unique())
densidades = sorted(df["Dens.Vac."].unique())

cmap = matplotlib.colormaps["plasma"].resampled(len(diametros))
color_map = {d: cmap(i) for i, d in enumerate(diametros)}

MARKERS = ["o", "D", "s", "^", "v", "P", "*", "X", "h", "<", ">", "p"]
marker_map = {dens: MARKERS[i % len(MARKERS)] for i, dens in enumerate(densidades)}

fs = plt.rcParams["legend.fontsize"]

leyenda_color = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[d], markersize=10, label=f"{d * 1e9:.2f}")
    for d in diametros
]
leyenda_dens = [
    Line2D([0], [0], marker=marker_map[dens], color="gray", markersize=8, linestyle="none", label=f"{int(dens)}")
    for dens in densidades
]

for col, titulo in zip(vecinos, titulos):
    fig, ax = plt.subplots(figsize=(12, 9))

    for d in diametros:
        for dens in densidades:
            sub = df[(df["Diametro (m)"] == d) & (df["Dens.Vac."] == dens)]
            if sub.empty:
                continue
            ax.scatter(
                sub[col],
                sub["V_reset_abs"],
                c=[color_map[d]] * len(sub),
                marker=marker_map[dens],
                alpha=0.6,
                edgecolors="none",
                s=80,
            )

    ax.set_xlabel(titulo)
    ax.set_ylabel("|V$_{reset}$| (V)")
    ax.set_title(f"|V$_{{reset}}$| vs {col}")

    leg1 = ax.legend(
        handles=leyenda_color,
        title="Diameter nm",
        fontsize=fs * 0.8,
        title_fontsize=fs * 0.85,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
    )
    ax.add_artist(leg1)
    ax.legend(
        handles=leyenda_dens,
        title=r"Density (vac./nm$^2$)",
        fontsize=fs * 0.8,
        title_fontsize=fs * 0.85,
        loc="upper left",
        bbox_to_anchor=(1.01, 0.5),
        markerscale=1.2,
        handlelength=1.5,
        handletextpad=0.5,
    )

    config_ax_IV(ax)
    fig.tight_layout()
    fig.savefig(f"Vreset_vs_{col}.pdf", dpi=150, bbox_inches="tight")
    plt.close(fig)


## V_RESET - HISTOGRAMA POR VACANTE DOMINANTE (AJUSTE CON GAUSSIANA) NO SE USA

In [ ]:
from scipy.stats import norm

vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
etiquetas = [r"$N_0$", r"$N_1$", r"$N_2$", r"$N_3$", r"$N_4$"]

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

# Dominante por fracción relativa
df["total"] = df[vecinos].sum(axis=1)
for col in vecinos:
    df[f"{col}_norm"] = df[col] / df["total"]
df["dominante"] = df[[f"{col}_norm" for col in vecinos]].idxmax(axis=1).str.replace("_norm", "", regex=False)

diametros = sorted(df["Diametro (m)"].unique())
cmap_vec = matplotlib.colormaps["viridis"].resampled(len(vecinos))
color_vec = {col: cmap_vec(i) for i, col in enumerate(vecinos)}

for d in diametros:
    sub = df[df["Diametro (m)"] == d]

    fig, ax = plt.subplots(figsize=(11, 7))
    plotted = False

    for col, etiq in zip(vecinos, etiquetas):
        datos = sub[sub["dominante"] == col]["V_reset_abs"].dropna()
        if len(datos) < 5:
            continue
        plotted = True
        color = color_vec[col]
        ax.hist(datos, bins=15, density=True, alpha=0.35, color=color, label=f"{etiq} (n={len(datos)})")
        mu, sigma = norm.fit(datos)
        xfit = np.linspace(datos.min(), datos.max(), 200)
        ax.plot(
            xfit, norm.pdf(xfit, mu, sigma), color=color, lw=2, label=f"{etiq}: $\\mu={mu:.3f}$, $\\sigma={sigma:.3f}$"
        )

    ax.set_xlabel("|V$_{reset}$| (V)")
    ax.set_ylabel("Probability density")
    ax.set_title(f"V$_{{reset}}$ by dominant vacancy type (relative) — {d * 1e9:.2f} nm")
    ax.legend(
        title="Dominant type",
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        fontsize=plt.rcParams["legend.fontsize"] * 0.8,
    )
    config_ax_IV(ax)
    fig.tight_layout()
    fig.savefig(f"Vreset_hist_dominante_{d * 1e9:.2f}nm.pdf", dpi=150, bbox_inches="tight")
    plt.close(fig)


## Ajuste para ver si lo compacto que es el filamento determina la tensión de reset ara cada diámetro (NO SE USA)

Este plot divide las simulaciones de un diámetro concreto en 4 grupos según la fracción de N_4 (vacantes completamente rodeadas):

Q1 — filamentos poco densos, muchos huecos: pocos N_4, muchas vacantes aisladas
Q2/Q3 — morfología intermedia
Q4 — filamentos compactos: muchos N_4, núcleo denso

Si las gaussianas están desplazadas — Q4 a voltajes más altos que Q1 — queda la compacidad del núcleo determina la estabilidad del estado LRS y el coste energético del reset.
Si se solapan → la compacidad no predice V_reset para ese diámetro.


In [ ]:
# NO se emplea

from scipy.stats import norm

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

df["total"] = df[["N_0", "N_1", "N_2", "N_3", "N_4"]].sum(axis=1)
df["N4_frac"] = df["N_4"] / df["total"]

diametros = sorted(df["Diametro (m)"].unique())
colores_q = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
etiq_q = ["diffuse", "semi-diffuse", "semi-compact", "compact"]

for d in diametros:
    sub = df[df["Diametro (m)"] == d].copy()
    sub["cuartil"] = pd.qcut(sub["N4_frac"], q=4, labels=[0, 1, 2, 3])

    fig, ax = plt.subplots(figsize=(11, 7))

    for q, color, etiq in zip([0, 1, 2, 3], colores_q, etiq_q):
        datos = sub[sub["cuartil"] == q]["V_reset_abs"].dropna()
        if len(datos) < 5:
            continue
        ax.hist(datos, bins=12, density=True, alpha=0.3, color=color, label=f"{etiq} (n={len(datos)})")
        mu, sigma = norm.fit(datos)
        xfit = np.linspace(datos.min(), datos.max(), 200)
        ax.plot(xfit, norm.pdf(xfit, mu, sigma), color=color, lw=2, label=f"$\\mu={mu:.3f}$, $\\sigma={sigma:.3f}$")

    ax.set_xlabel("|V$_{reset}$| (V)")
    ax.set_ylabel("Probability density")
    ax.set_title(f"V$_{{reset}}$ distribution by compactness — {d * 1e9:.2f} nm")
    ax.set_title(f"CF compactness determine $V_{{reset}}$ — {d*1e9:.2f} nm")
    ax.legend(
        title="Filament compactness",
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        fontsize=plt.rcParams["legend.fontsize"] * 0.8,
    )
    config_ax_IV(ax)
    fig.tight_layout()
    fig.savefig(f"Vreset_dist_by_compactness_{d * 1e9:.2f}nm.pdf", dpi=150, bbox_inches="tight")
    plt.close(fig)


## V_reset en función de la fracción de N_4 (NO SE USA Y NO TIENE SENTIDO LA DISTRIBUCIÓN DE DISPERSION DE V_RESET PUESTA ARRIBA)

In [ ]:
from scipy.stats import norm

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()
df["total"] = df[["N_0", "N_1", "N_2", "N_3", "N_4"]].sum(axis=1)
df["N4_frac"] = df["N_4"] / df["total"]

diametros = sorted(df["Diametro (m)"].unique())
cmap = matplotlib.colormaps["plasma"].resampled(len(diametros))
color_map = {d: cmap(i) for i, d in enumerate(diametros)}

for d in diametros:
    sub = df[df["Diametro (m)"] == d].dropna(subset=["V_reset_abs", "N4_frac"])
    x = sub["V_reset_abs"].values
    y = sub["N4_frac"].values
    color = color_map[d]

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], hspace=0.05, wspace=0.05)

    ax_main = fig.add_subplot(gs[1, 0])
    ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

    # Scatter central
    ax_main.scatter(x, y, c=[color] * len(x), alpha=0.5, edgecolors="none", s=40)
    ax_main.set_xlabel("|V$_{reset}$| (V)")
    ax_main.set_ylabel(r"$N_4$ fraction  $N_4\,/\,N_{total}$")
    config_ax_IV(ax_main)

    # Histograma + gaussiana superior (V_reset)
    ax_top.hist(x, bins=20, density=True, color=color, alpha=0.5)
    mu, sigma = norm.fit(x)
    xfit = np.linspace(x.min(), x.max(), 300)
    ax_top.plot(xfit, norm.pdf(xfit, mu, sigma), color="black", lw=1.8, label=f"$\\mu={mu:.3f}$\n$\\sigma={sigma:.3f}$")
    ax_top.legend(fontsize=8)
    ax_top.set_ylabel("Density")
    plt.setp(ax_top.get_xticklabels(), visible=False)
    config_ax_IV(ax_top)

    # Histograma + gaussiana derecha (N4_frac)
    ax_right.hist(y, bins=20, density=True, color=color, alpha=0.5, orientation="horizontal")
    mu2, sigma2 = norm.fit(y)
    yfit = np.linspace(y.min(), y.max(), 300)
    ax_right.plot(
        norm.pdf(yfit, mu2, sigma2), yfit, color="black", lw=1.8, label=f"$\\mu={mu2:.3f}$\n$\\sigma={sigma2:.3f}$"
    )
    ax_right.legend(fontsize=8)
    ax_right.set_xlabel("Density")
    plt.setp(ax_right.get_yticklabels(), visible=False)
    config_ax_IV(ax_right)

    fig.suptitle(f"V$_{{reset}}$ vs $N_4$ fraction — {d * 1e9:.2f} nm", y=1.01)
    fig.savefig(f"jointplot_Vreset_N4_{d * 1e9:.2f}nm.pdf", dpi=150, bbox_inches="tight")
    plt.close(fig)


## Vreset vs vacancy type fractions POR DIÁMETRO, NO TIENE MUCHO SENTIDO PONER LA DISTRIBUCIÓN DE V_RESET ENCIMA DE CADA PLOT, YA Q SIEMPRE ES LA MISMA (NO SE USA)

In [ ]:
# Fracción de ocupacion para cada vacante a un mismo diámetro, son 5 plot para los 5 tipo s de vacantes y tmb se representa su histograma

from scipy.stats import norm

vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
etiquetas = [r"$N_0$", r"$N_1$", r"$N_2$", r"$N_3$", r"$N_4$"]

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()
df["total"] = df[vecinos].sum(axis=1)
for col in vecinos:
    df[f"{col}_frac"] = df[col] / df["total"]

diametros = sorted(df["Diametro (m)"].unique())
cmap_vec = matplotlib.colormaps["viridis"].resampled(len(vecinos))
color_vec = {col: cmap_vec(i) for i, col in enumerate(vecinos)}

for d in diametros:
    sub = df[df["Diametro (m)"] == d].dropna(subset=["V_reset_abs"])

    fig = plt.figure(figsize=(36, 12))
    # Para cada Nk: columna de 3 filas (hist_top, scatter, hist_right)
    # Usamos GridSpec con 3 filas × (5×2) columnas
    outer = fig.add_gridspec(1, 5, wspace=0.35)

    for idx, (col, etiq) in enumerate(zip(vecinos, etiquetas)):
        frac_col = f"{col}_frac"
        x = sub["V_reset_abs"].values
        y = sub[frac_col].values
        color = color_vec[col]

        inner = outer[idx].subgridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], hspace=0.05, wspace=0.05)

        ax_main = fig.add_subplot(inner[1, 0])
        ax_top = fig.add_subplot(inner[0, 0], sharex=ax_main)
        ax_right = fig.add_subplot(inner[1, 1], sharey=ax_main)

        # Scatter
        ax_main.scatter(x, y, c=[color] * len(x), alpha=0.5, edgecolors="none", s=25)
        ax_main.set_xlabel("|V$_{reset}$| (V)")
        if idx == 0:
            ax_main.set_ylabel("Fraction")
        config_ax_IV(ax_main)

        # Hist top — V_reset
        ax_top.hist(x, bins=20, density=True, color=color, alpha=0.5)
        mu, sigma = norm.fit(x)
        xfit = np.linspace(x.min(), x.max(), 300)
        ax_top.plot(xfit, norm.pdf(xfit, mu, sigma), color="black", lw=1.5)
        ax_top.set_title(
            f"{etiq}\n$\\mu_{{V}}={mu:.3f}$, $\\sigma_{{V}}={sigma:.3f}$", fontsize=plt.rcParams["axes.titlesize"] * 0.8
        )
        plt.setp(ax_top.get_xticklabels(), visible=False)
        config_ax_IV(ax_top)

        # Hist right — fracción Nk
        ax_right.hist(y, bins=20, density=True, color=color, alpha=0.5, orientation="horizontal")
        mu2, sigma2 = norm.fit(y)
        yfit = np.linspace(y.min(), y.max(), 300)
        ax_right.plot(norm.pdf(yfit, mu2, sigma2), yfit, color="black", lw=1.5)
        ax_right.set_title(f"$\\mu={mu2:.3f}$\n$\\sigma={sigma2:.3f}$", fontsize=plt.rcParams["axes.titlesize"] * 0.7)
        plt.setp(ax_right.get_yticklabels(), visible=False)
        config_ax_IV(ax_right)

    fig.suptitle(f"V$_{{reset}}$ vs vacancy type fractions — {d * 1e9:.2f} nm", y=1.02)
    fig.savefig(f"jointplot_Vreset_allN_{d * 1e9:.2f}nm.pdf", dpi=150, bbox_inches="tight")
    plt.close(fig)


## V_RESET VS CANTIDAD Y TIPO DE VACANTE. A LA DERECHA SE MUESTRA EL HISTOGRAMA CON GAUSSIANAS DE LA DISTRIBUCIÓN DE VACANTES Y ENCIMA SE MUESTRA EL HISTOGRAMA DE V_RESET PARA esa densidad y diámetro (NO SE USA)

In [ ]:
# NO se usa

from scipy.stats import norm

vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
etiquetas = [r"$N_0$", r"$N_1$", r"$N_2$", r"$N_3$", r"$N_4$"]

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

diametros = sorted(df["Diametro (m)"].unique())
densidades = sorted(df["Dens.Vac."].unique(), reverse=True)

cmap_d = matplotlib.colormaps["plasma"].resampled(len(diametros))
color_map = {d: cmap_d(i) for i, d in enumerate(diametros)}

cmap_vec = matplotlib.colormaps["viridis"].resampled(len(vecinos))
color_vec = {col: cmap_vec(i) for i, col in enumerate(vecinos)}

fs = plt.rcParams["legend.fontsize"]

leyenda_color = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_vec[col], markersize=10, label=etiq)
    for col, etiq in zip(vecinos, etiquetas)
]

for d in diametros:
    for dens in densidades:
        sub = df[(df["Diametro (m)"] == d) & (df["Dens.Vac."] == dens)].dropna(subset=["V_reset_abs"])
        x = sub["V_reset_abs"].values
        if len(x) < 5:
            continue

        fig = plt.figure(figsize=(14, 10))
        gs = fig.add_gridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], hspace=0.05, wspace=0.05, right=0.72)

        ax_main = fig.add_subplot(gs[1, 0])
        ax_top = fig.add_subplot(gs[0, 0], sharex=ax_main)
        ax_right = fig.add_subplot(gs[1, 1])

        # ── Scatter ───────────────────────────────────────────────────────────
        for col in vecinos:
            yi = sub[col].values
            ax_main.scatter(x, yi, c=[color_vec[col]] * len(x), alpha=0.5, edgecolors="none", s=40)
        ax_main.set_xlabel("|V$_{reset}$| (V)")
        ax_main.set_ylabel(r"$N_k$")
        ax_main.legend(
            handles=leyenda_color,
            title="Type",
            fontsize=fs,
            title_fontsize=fs,
            loc="upper left",
            bbox_to_anchor=(1.3, 0.62),
            markerscale=1.2,
        )
        config_ax_IV(ax_main)

        # ── Hist top — V_reset ────────────────────────────────────────────────
        mu, sigma = norm.fit(x)
        n_bins = min(15, max(5, len(x) // 3))
        ax_top.hist(x, bins=n_bins, density=True, color=color_map[d], alpha=0.5)
        xfit = np.linspace(mu - 3 * sigma, mu + 3 * sigma, 300)
        ax_top.plot(
            xfit, norm.pdf(xfit, mu, sigma), color="black", lw=2.5, label=f"$\\mu={mu:.3f}$, $\\sigma={sigma:.3f}$"
        )
        ax_top.legend(fontsize=fs, loc="upper left", bbox_to_anchor=(1, 1.0))
        ax_top.set_ylabel("Density")
        plt.setp(ax_top.get_xticklabels(), visible=False)
        config_ax_IV(ax_top)

        # ── Hist right — Nk por tipo ──────────────────────────────────────────
        for col, etiq in zip(vecinos, etiquetas):
            y = sub[col].values
            color = color_vec[col]
            n_bins_k = min(15, max(5, len(y) // 3))
            ax_right.hist(y, bins=n_bins_k, density=True, color=color, alpha=0.4, orientation="horizontal")
            mu2, sigma2 = norm.fit(y)
            yfit = np.linspace(mu2 - 3 * sigma2, mu2 + 3 * sigma2, 200)
            ax_right.plot(norm.pdf(yfit, mu2, sigma2), yfit, color=color, lw=1.5, label=etiq)
        ax_right.legend(fontsize=fs, loc="upper left", bbox_to_anchor=(1.07, 1.0))
        ax_right.set_xlabel("Density")
        plt.setp(ax_right.get_yticklabels(), visible=False)
        config_ax_IV(ax_right)

        fig.suptitle(f"V$_{{reset}}$ vs $N_k$ — {d * 1e9:.2f} nm · density={int(dens)}", y=0.98)
        fig.subplots_adjust(top=0.93)
        fig.savefig(f"jointplot_{d * 1e9:.2f}nm_dens{int(dens)}.pdf", dpi=150, bbox_inches="tight")
        plt.close(fig)


## V_RESET VS CANTIDAD Y TIPO DE VACANTE. A LA DERECHA SE MUESTRA EL HISTOGRAMA CON GAUSSIANAS DE LA DISTRIBUCIÓN DE VACANTES esa densidad y diámetro (NO SE USA)

In [ ]:
from scipy.stats import norm

vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
etiquetas = [r"$N_0$", r"$N_1$", r"$N_2$", r"$N_3$", r"$N_4$"]

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

diametros = sorted(df["Diametro (m)"].unique())
densidades = sorted(df["Dens.Vac."].unique(), reverse=True)

cmap_d = matplotlib.colormaps["plasma"].resampled(len(diametros))
color_map = {d: cmap_d(i) for i, d in enumerate(diametros)}

cmap_vec = matplotlib.colormaps["viridis"].resampled(len(vecinos))
color_vec = {col: cmap_vec(i) for i, col in enumerate(vecinos)}

fs = plt.rcParams["legend.fontsize"]

leyenda_color = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_vec[col], markersize=10, label=etiq)
    for col, etiq in zip(vecinos, etiquetas)
]

for d in diametros:
    for dens in densidades:
        sub = df[(df["Diametro (m)"] == d) & (df["Dens.Vac."] == dens)].dropna(subset=["V_reset_abs"])
        x = sub["V_reset_abs"].values
        if len(x) < 5:
            continue

        fig = plt.figure(figsize=(16, 9))
        gs = fig.add_gridspec(1, 2, width_ratios=[4, 1], wspace=0.15, right=0.72)

        ax_main = fig.add_subplot(gs[0, 0])
        ax_right = fig.add_subplot(gs[0, 1])

        # ── Scatter ───────────────────────────────────────────────────────────
        for col in vecinos:
            yi = sub[col].values
            ax_main.scatter(x, yi, c=[color_vec[col]] * len(x), alpha=0.5, edgecolors="none", s=40)
        ax_main.set_xlabel("|V$_{reset}$| (V)")
        ax_main.set_ylabel(r"$N_k$ (vacancy count)")
        ax_main.legend(
            handles=leyenda_color,
            title="Type",
            fontsize=fs,
            title_fontsize=fs,
            loc="upper left",
            bbox_to_anchor=(1.36, 0.65),
            markerscale=1.2,
        )
        config_ax_IV(ax_main)

        # ── Hist right — distribución del número de cada Nk ──────────────────
        for col, etiq in zip(vecinos, etiquetas):
            y = sub[col].values
            color = color_vec[col]
            n_bins_k = min(15, max(5, len(y) // 3))
            ax_right.hist(y, bins=n_bins_k, density=True, color=color, alpha=0.4, orientation="horizontal")
            mu2, sigma2 = norm.fit(y)
            yfit = np.linspace(mu2 - 3 * sigma2, mu2 + 3 * sigma2, 200)
            ax_right.plot(norm.pdf(yfit, mu2, sigma2), yfit, color=color, lw=1.5, label=f"{etiq}")
        ax_right.legend(fontsize=fs, loc="upper left", bbox_to_anchor=(1.07, 1.0))
        ax_right.set_xlabel("Density")
        ax_right.set_ylabel(r"$N_k$ (vacancy count)")
        plt.setp(ax_right.get_yticklabels(), visible=False)
        config_ax_IV(ax_right)

        fig.suptitle(f"V$_{{reset}}$ vs $N_k$ — {d * 1e9:.2f} nm · density={int(dens)} ($\\text{{vacancy}}/\\text{{nm}}^2$)", y=0.98)
        fig.savefig(f"jointplot_{d * 1e9:.2f}nm_dens{int(dens)}_no_V_reset_hist.pdf", dpi=150, bbox_inches="tight")
        plt.close(fig)


## V_RESET VS CANTIDAD Y TIPO DE VACANTE. A LA DERECHA SE MUESTRA EL HISTOGRAMA CON GAUSSIANAS el recuento DE VACANTES para esa densidad y diámetro (NO SE USA)


In [ ]:
from scipy.stats import norm

vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
etiquetas = [r"$N_0$", r"$N_1$", r"$N_2$", r"$N_3$", r"$N_4$"]

df = pd.read_csv(filename, sep="\t")
df["V_reset (V)"] = pd.to_numeric(df["V_reset (V)"], errors="coerce")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")
df["V_reset_abs"] = df["V_reset (V)"].abs()

diametros = sorted(df["Diametro (m)"].unique())
densidades = sorted(df["Dens.Vac."].unique(), reverse=True)

cmap_d = matplotlib.colormaps["plasma"].resampled(len(diametros))
color_map = {d: cmap_d(i) for i, d in enumerate(diametros)}

cmap_vec = matplotlib.colormaps["viridis"].resampled(len(vecinos))
color_vec = {col: cmap_vec(i) for i, col in enumerate(vecinos)}

fs = plt.rcParams["legend.fontsize"]

leyenda_color = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_vec[col], markersize=10, label=etiq)
    for col, etiq in zip(vecinos, etiquetas)
]

for d in diametros:
    for dens in densidades:
        sub = df[(df["Diametro (m)"] == d) & (df["Dens.Vac."] == dens)].dropna(subset=["V_reset_abs"])
        x = sub["V_reset_abs"].values
        if len(x) < 5:
            continue

        fig = plt.figure(figsize=(16, 9))
        gs = fig.add_gridspec(1, 2, width_ratios=[4, 1], wspace=0.25, right=0.72)

        ax_main = fig.add_subplot(gs[0, 0])
        ax_right = fig.add_subplot(gs[0, 1])

        # ── Scatter ───────────────────────────────────────────────────────────
        for col in vecinos:
            yi = sub[col].values
            ax_main.scatter(x, yi, c=[color_vec[col]] * len(x), alpha=0.5, edgecolors="none", s=40)
        ax_main.set_xlabel("|V$_{reset}$| (V)")
        ax_main.set_ylabel(r"$N_k$ (vacancy count)")
        ax_main.legend(
            handles=leyenda_color,
            title="Type",
            fontsize=fs,
            title_fontsize=fs,
            loc="upper left",
            bbox_to_anchor=(1.45, 0.65),
            markerscale=1.2,
        )
        config_ax_IV(ax_main)

        # ── Bar right — conteo total por tipo ────────────────────────────────
        totales = [sub[col].sum() for col in vecinos]
        colores = [color_vec[col] for col in vecinos]
        ax_right.barh(etiquetas, totales, color=colores, alpha=0.8, edgecolor="black", linewidth=0.5)
        ax_right.set_xlabel("Total count")
        ax_right.set_ylabel(r"$N_k$")
        plt.setp(ax_right.get_yticklabels(), visible=True)
        config_ax_IV(ax_right)

        fig.suptitle(
            f"V$_{{reset}}$ vs $N_k$ — {d * 1e9:.2f} nm · density={int(dens)} ($\\text{{vacancy}}/\\text{{nm}}^2$)",
            y=0.98,
        )
        fig.savefig(f"jointplot_{d * 1e9:.2f}nm_num_vac_dens{int(dens)}.pdf", dpi=150, bbox_inches="tight")
        plt.close(fig)


## Histograma de recuento de tipo de vacantes / num simulaciones para cada densidad y diámetro

In [24]:
df = pd.read_csv(filename, sep="\t")
df["Diametro (m)"] = pd.to_numeric(df["Diametro (m)"], errors="coerce")

vecinos = ["N_0", "N_1", "N_2", "N_3", "N_4"]
etiquetas = [r"$N_0$", r"$N_1$", r"$N_2$", r"$N_3$", r"$N_4$"]

diametros = sorted(df["Diametro (m)"].unique())
densidades = sorted(df["Dens.Vac."].unique(), reverse=True)

cmap_vec = matplotlib.colormaps["viridis"].resampled(len(vecinos))
color_vec = {col: cmap_vec(i) for i, col in enumerate(vecinos)}

for d in diametros:
    for dens in densidades:
        sub = df[(df["Diametro (m)"] == d) & (df["Dens.Vac."] == dens)]
        if len(sub) < 1:
            continue

        n_sims = len(sub)
        # Número de veces que aparece cada tipo / n_sims
        fracs = [sub[col].sum() / n_sims for col in vecinos]

        fig, ax = plt.subplots(figsize=(8, 6))
        bars = ax.bar(
            etiquetas,
            fracs,
            color=[cmap_vec(i) for i in range(len(vecinos))],
            alpha=0.85,
            edgecolor="black",
            linewidth=0.6,
        )

        ax.set_xlabel("Vacancy type")
        ax.set_ylabel(r"$\sum N_k \;/\; N_{sims}$")
        ax.set_title(f"Vacancy frequency — {d * 1e9:.2f} nm · density={int(dens)}")
        config_ax_IV(ax)
        fig.tight_layout()
        fig.savefig(f"vacancy_freq_{d * 1e9:.2f}nm_dens{int(dens)}.pdf", dpi=150, bbox_inches="tight")
        plt.close(fig)
